# EcoSignal Coverage v3 — inference and generalization repair

Run the corrected workflow below in a fresh Colab runtime. This notebook contains **no new trained weights or new model accuracy claims**. Every original output is preserved in the historical appendix; those results do not validate this revision.

The uploaded run used 99 pairs (64 train / 14 validation / 21 test), threshold 0.55, test mean IoU 0.74265 and coverage MAE 4.58854 percentage points. Its phone-image prediction was 0.61%. These differ from the earlier chat's run. High mean pixel certainty (94.76%) did not imply correctness.

**Diagnosis:** RGB normalization is already broadly consistent. The original MobileNetV2 graph can retain its upstream Rescaling via `get_source_inputs`; a bypass is not established. The revision makes that graph explicit. Binary-mask decoding with `>0` can corrupt compressed/antialiased/color labels, but actual label corruption cannot be proven without the source masks. Random image splitting does not protect against related photos. A new fine-tuning checkpoint callback could overwrite a better head checkpoint. Field texture, distance, lighting and growth form differ from much of the visible holdout data; this is the strongest current explanation, not a proven sole cause.

**Needed for a real repair:** upload Images.zip and Masks.zip; review label overlays; supply source/site groups where known; include representative phone photos and hard negatives in development data. Keep separate labeled field photos untouched for final evaluation. Do not tune against the old repeatedly inspected test set or the final field holdout.


In [ ]:
# Colab includes TensorFlow. Avoid upgrading the whole scientific stack or deleting packages.
import json, random, hashlib, warnings, zipfile, uuid
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print('TensorFlow', tf.__version__, 'GPU', tf.config.list_physical_devices('GPU'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/EcoSignalCoverage')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUTPUT_DIR = PROJECT_DIR / 'training_output' / ('coverage_v3_' + RUN_ID)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
IMAGE_SIZE, BATCH_SIZE = 320, 4
HEAD_EPOCHS, FINE_TUNE_EPOCHS = 18, 10
VAL_FRACTION, TEST_FRACTION, SEED = 0.15, 0.20, 42
# Explicit binary-label contract: use (0,1) for indexed masks; do not guess foreground.
MASK_BACKGROUND, MASK_FOREGROUND = 0, 255
# 'display': masks were drawn on EXIF-oriented images. 'raw': masks use raw sensor pixels.
MASK_COORDINATES = 'display'
SOURCE_GROUPS_CSV = PROJECT_DIR / 'source_groups.csv' # columns: stem, group_id
EXTENSIONS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.avif'}
keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Fresh extraction avoids silently retaining files from a previous dataset.
def extract_dataset(archive, folder):
    archive, folder = Path(archive), Path(folder)
    if not archive.is_file():
        raise FileNotFoundError(f'Upload {archive} first')
    folder.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            target = (folder / item.filename).resolve()
            if not target.is_relative_to(folder.resolve()):
                raise ValueError('Unsafe archive path')
        z.extractall(folder)
    return folder
IMAGE_DIR = extract_dataset('/content/Images.zip', '/content/coverage_images_' + RUN_ID)
MASK_DIR = extract_dataset('/content/Masks.zip', '/content/coverage_masks_' + RUN_ID)

def files_by_stem(folder):
    found = {}
    for p in sorted(Path(folder).rglob('*')):
        if not p.is_file() or p.suffix.lower() not in EXTENSIONS or '__MACOSX' in p.parts:
            continue
        if p.stem in found:
            raise ValueError(f'Duplicate stem {p.stem}: {found[p.stem]} and {p}')
        found[p.stem] = p
    return found
images, masks = files_by_stem(IMAGE_DIR), files_by_stem(MASK_DIR)
if set(images) != set(masks):
    raise ValueError(f'Unpaired data. Missing masks: {set(images)-set(masks)}; missing images: {set(masks)-set(images)}')
if len(images) < 40:
    raise ValueError('Too few pairs for this three-way evaluation')
pairs = pd.DataFrame([dict(stem=s, image_path=str(images[s]), mask_path=str(masks[s])) for s in sorted(images)])


In [ ]:
def load_rgb(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert('RGB')).copy()

def decode_binary_mask(path):
    if Path(path).suffix.lower() in {'.jpg','.jpeg','.avif','.webp'}:
        raise ValueError('Use lossless binary PNG/TIFF masks; export from original annotations')
    with Image.open(path) as im:
        if im.mode not in {'1','L','P','I','I;16','RGB','RGBA'}:
            raise ValueError(f'Unsupported mask mode {im.mode}: {path}')
        a = np.asarray(im).copy() # preserve palette indices, not palette luminance
    if a.ndim == 3:
        if a.shape[-1] == 4 and not np.all(a[...,3] == 255):
            raise ValueError('Transparent mask needs explicit conversion from annotation source')
        if not (np.array_equal(a[...,0],a[...,1]) and np.array_equal(a[...,1],a[...,2])):
            raise ValueError('Color masks need an explicit class-to-lichen mapping')
        a = a[...,0]
    allowed = {MASK_BACKGROUND, MASK_FOREGROUND}
    if not set(np.unique(a)).issubset(allowed):
        raise ValueError(f'Mask values {np.unique(a)[:20]} violate {allowed}: {path}')
    return (a == MASK_FOREGROUND).astype(np.float32)

def load_pair_arrays(image_path, mask_path):
    rgb, mask = load_rgb(image_path), decode_binary_mask(mask_path)
    if MASK_COORDINATES == 'raw':
        with Image.open(image_path) as im:
            orientation = im.getexif().get(274, 1)
        operations = {2:Image.Transpose.FLIP_LEFT_RIGHT,3:Image.Transpose.ROTATE_180,
            4:Image.Transpose.FLIP_TOP_BOTTOM,5:Image.Transpose.TRANSPOSE,
            6:Image.Transpose.ROTATE_270,7:Image.Transpose.TRANSVERSE,8:Image.Transpose.ROTATE_90}
        if orientation in operations:
            mask = np.asarray(Image.fromarray(mask).transpose(operations[orientation]))
    elif MASK_COORDINATES != 'display':
        raise ValueError('Invalid MASK_COORDINATES')
    if rgb.shape[:2] != mask.shape:
        raise ValueError(f'Image/mask geometry mismatch: {image_path}; {rgb.shape} vs {mask.shape}')
    return rgb, mask

def coverage_bin(value):
    if value == 0: return 'negative'
    if value < 10: return 'very_low'
    if value < 30: return 'low'
    if value < 60: return 'medium'
    return 'high'

def dhash(rgb):
    a = np.asarray(Image.fromarray(rgb).convert('L').resize((9,8)))
    return np.packbits(a[:,1:] > a[:,:-1]).tobytes().hex()

audit = []
for r in pairs.itertuples():
    rgb, mask = load_pair_arrays(r.image_path, r.mask_path)
    resized_cov=float(tf.image.resize(mask[...,None],[IMAGE_SIZE,IMAGE_SIZE],method='nearest').numpy().mean()*100)
    audit.append(dict(stem=r.stem, width=rgb.shape[1], height=rgb.shape[0],
        coverage_percent=float(mask.mean()*100),
        coverage_resized_percent=resized_cov,resize_coverage_delta=resized_cov-float(mask.mean()*100),
        pixel_sha256=hashlib.sha256(str(rgb.shape).encode()+rgb.tobytes()).hexdigest(),
        dhash=dhash(rgb)))
pairs = pairs.merge(pd.DataFrame(audit), on='stem', validate='one_to_one')
pairs['coverage_bin'] = pairs.coverage_percent.map(coverage_bin)
pairs.to_csv(OUTPUT_DIR/'dataset_audit.csv', index=False)
display(pairs.groupby('coverage_bin').size())
# Mandatory human inspection: binary correctness does not prove annotation completeness.
for r in pairs.sample(min(12,len(pairs)),random_state=SEED).itertuples():
    rgb, mask = load_pair_arrays(r.image_path,r.mask_path)
    fig, ax = plt.subplots(1,2,figsize=(8,3))
    ax[0].imshow(rgb); ax[1].imshow(rgb)
    ax[1].imshow(np.ma.masked_where(mask==0,mask),cmap='autumn',alpha=.45,vmin=0,vmax=1)
    fig.suptitle(f'{r.stem}: {r.coverage_percent:.2f}%'); plt.show()


Review overlays for missing growth forms, white/gray crustose patches, coarse polygons and inverse labels. Correct source annotations, never predictions. A mask that labels only prominent yellow or leafy patches teaches the model to ignore other lichen. Set `LABEL_REVIEW_COMPLETE=True` below after review. For rigorous site/specimen evaluation, fill source_groups.csv before splitting; automatic hashes cannot identify every related photograph.


In [ ]:
LABEL_REVIEW_COMPLETE = False
if not LABEL_REVIEW_COMPLETE:
    raise RuntimeError('Review the overlays and mask contract above, then set LABEL_REVIEW_COMPLETE=True')

def assign_groups(frame, source_groups=None):
    frame = frame.reset_index(drop=True).copy()
    parent = list(range(len(frame)))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def union(i,j): parent[root(j)] = root(i)
    if source_groups is not None:
        if source_groups.stem.duplicated().any() or set(source_groups.stem) != set(frame.stem):
            raise ValueError('source_groups.csv needs exactly one row per dataset stem')
        if source_groups.group_id.isna().any() or source_groups.group_id.astype(str).str.strip().eq('').any():
            raise ValueError('Missing source group IDs')
        mapping = source_groups.set_index('stem').group_id.to_dict()
    else:
        mapping = {}
        warnings.warn('No source groups: hashes reduce duplicate leakage but cannot establish site independence')
    similar = []
    for i in range(len(frame)):
        for j in range(i):
            a,b = frame.iloc[i],frame.iloc[j]
            distance = (int(a.dhash,16)^int(b.dhash,16)).bit_count()
            if a.pixel_sha256 == b.pixel_sha256 or distance <= 4 or (mapping and mapping[a.stem] == mapping[b.stem]):
                union(i,j)
                similar.append(dict(stem_a=a.stem,stem_b=b.stem,dhash_distance=distance))
    frame['group_id'] = [f'g{root(i):04d}' for i in range(len(frame))]
    return frame, pd.DataFrame(similar, columns=['stem_a','stem_b','dhash_distance'])

source_groups = pd.read_csv(SOURCE_GROUPS_CSV,dtype=str) if SOURCE_GROUPS_CSV.exists() else None
if source_groups is None:
    pd.DataFrame({'stem':pairs.stem,'group_id':''}).to_csv(OUTPUT_DIR/'source_groups_template.csv',index=False)
pairs, similar = assign_groups(pairs,source_groups)
similar.to_csv(OUTPUT_DIR/'related_image_candidates.csv',index=False)

def group_split(frame, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED):
    groups = frame.group_id.unique()
    if len(groups) < 5: raise ValueError('Too few independent groups')
    rng = np.random.default_rng(seed)
    bins = sorted(frame.coverage_bin.unique())
    target = np.array([1-val_fraction-test_fraction,val_fraction,test_fraction])
    counts = pd.crosstab(frame.group_id,frame.coverage_bin).reindex(index=groups,columns=bins,fill_value=0).to_numpy()
    totals = counts.sum(axis=0)
    nt = max(1,round(len(groups)*test_fraction)); nv = max(1,round(len(groups)*val_fraction))
    best = None
    # Optimize label balance only; no model predictions or test performance used.
    for _ in range(1000):
        perm = rng.permutation(len(groups))
        if nt+nv >= len(groups): continue
        indexes = [perm[nt+nv:],perm[nt:nt+nv],perm[:nt]]
        split_counts = np.array([counts[ix].sum(axis=0) for ix in indexes])
        score = np.abs(split_counts.sum(axis=1)/len(frame)-target).sum()
        score += np.abs(split_counts/totals-target[:,None]).sum()/len(bins)
        if best is None or score < best[0]: best = (score,indexes)
    if best is None: raise ValueError('Invalid split fractions')
    train,val,test = [frame[frame.group_id.isin(groups[ix])].copy() for ix in best[1]]
    assert not (set(train.group_id)&set(val.group_id) or set(train.group_id)&set(test.group_id) or set(val.group_id)&set(test.group_id))
    assert len(train)+len(val)+len(test)==len(frame)
    return train,val,test
train_df,val_df,test_df = group_split(pairs)
manifest = {name:df[['stem','group_id','pixel_sha256']].to_dict('records') for name,df in [('train',train_df),('validation',val_df),('test',test_df)]}
(OUTPUT_DIR/'split_manifest.json').write_text(json.dumps(manifest,indent=2))
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    print(name,len(frame)); display(frame.coverage_bin.value_counts())


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def preprocess_rgb(rgb):
    # Single contract for training, validation, holdout and uploaded field photographs.
    image = tf.convert_to_tensor(rgb, dtype=tf.uint8)
    image = tf.cast(image,tf.float32)/255.0
    return tf.image.resize(image,[IMAGE_SIZE,IMAGE_SIZE],method='bilinear',antialias=True)

def read_image(path):
    return preprocess_rgb(load_rgb(path))

def _load_numpy(ip,mp):
    def path(v): return v.numpy().decode('utf-8')
    rgb,mask = load_pair_arrays(path(ip),path(mp))
    return rgb, mask[...,None]

def load_pair(ip,mp):
    rgb,mask = tf.py_function(_load_numpy,[ip,mp],[tf.uint8,tf.float32])
    rgb.set_shape([None,None,3]); mask.set_shape([None,None,1])
    image = preprocess_rgb(rgb)
    mask = tf.image.resize(mask,[IMAGE_SIZE,IMAGE_SIZE],method='nearest')
    return image,mask

def augment(image,mask):
    # Joint crop and geometry, then image-only photometric changes.
    joined = tf.concat([image,mask],axis=-1)
    side = tf.random.uniform([],int(IMAGE_SIZE*.65),IMAGE_SIZE+1,dtype=tf.int32)
    joined = tf.image.random_crop(joined,[side,side,4])
    joined = tf.image.random_flip_left_right(joined)
    joined = tf.image.random_flip_up_down(joined)
    joined = tf.image.rot90(joined,tf.random.uniform([],0,4,dtype=tf.int32))
    image = tf.image.resize(joined[...,:3],[IMAGE_SIZE,IMAGE_SIZE],antialias=True)
    mask = tf.image.resize(joined[...,3:],[IMAGE_SIZE,IMAGE_SIZE],method='nearest')
    image = tf.image.random_brightness(image,.15)
    image = tf.image.random_contrast(image,.7,1.3)
    image = tf.image.random_saturation(image,.7,1.3)
    image = tf.image.adjust_gamma(tf.clip_by_value(image,0,1),tf.random.uniform([],.8,1.2))
    image += tf.random.normal(tf.shape(image),stddev=.01)
    return tf.clip_by_value(image,0,1),mask

def make_dataset(df,training=False):
    ds = tf.data.Dataset.from_tensor_slices((df.image_path.values,df.mask_path.values))
    if training: ds = ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    ds = ds.map(load_pair,num_parallel_calls=AUTOTUNE)
    if training: ds = ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
train_ds,val_ds,test_ds = make_dataset(train_df,True),make_dataset(val_df),make_dataset(test_df)


In [ ]:
SMOOTH = 1e-6

def dice_coefficient(y_true,y_pred):
    yt = tf.reshape(tf.cast(y_true,tf.float32),[tf.shape(y_true)[0],-1])
    yp = tf.reshape(tf.cast(y_pred,tf.float32),[tf.shape(y_pred)[0],-1])
    inter = tf.reduce_sum(yt*yp,axis=1)
    denom = tf.reduce_sum(yt,axis=1)+tf.reduce_sum(yp,axis=1)
    return tf.reduce_mean((2*inter+SMOOTH)/(denom+SMOOTH))

def soft_iou(y_true,y_pred):
    yt = tf.reshape(tf.cast(y_true,tf.float32),[tf.shape(y_true)[0],-1])
    yp = tf.reshape(tf.cast(y_pred,tf.float32),[tf.shape(y_pred)[0],-1])
    inter = tf.reduce_sum(yt*yp,axis=1)
    union = tf.reduce_sum(yt,axis=1)+tf.reduce_sum(yp,axis=1)-inter
    return tf.reduce_mean((inter+SMOOTH)/(union+SMOOTH))

bce = keras.losses.BinaryCrossentropy()

def combined_loss(y_true,y_pred):
    return 0.5*bce(y_true,y_pred)+0.5*(1-dice_coefficient(y_true,y_pred))


In [ ]:
def decoder_block(x,skip,filters,name):
    x = layers.Conv2DTranspose(filters,3,strides=2,padding="same",
                               use_bias=False,name=name+"_up")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    if skip is not None:
        x = layers.Concatenate()([x,skip])
    for _ in range(2):
        x = layers.Conv2D(filters,3,padding="same",use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x

def build_model(encoder_weights="imagenet"):
    inputs = keras.Input((IMAGE_SIZE,IMAGE_SIZE,3))
    x0 = layers.Rescaling(2.0,offset=-1.0,name="mobilenet_scaling")(inputs)

    base = keras.applications.MobileNetV2(
        input_shape=(IMAGE_SIZE,IMAGE_SIZE,3), include_top=False, weights=encoder_weights
    )

    skip_names = [
        "block_1_expand_relu",
        "block_3_expand_relu",
        "block_6_expand_relu",
        "block_13_expand_relu",
    ]

    outputs = [base.get_layer(n).output for n in skip_names]
    outputs.append(base.get_layer("block_16_project").output)
    encoder = keras.Model(base.input,outputs,name="encoder")
    encoder.trainable = False

    s1,s2,s3,s4,x = encoder(x0,training=False)
    x = decoder_block(x,s4,256,"d1")
    x = decoder_block(x,s3,128,"d2")
    x = decoder_block(x,s2,64,"d3")
    x = decoder_block(x,s1,32,"d4")
    x = decoder_block(x,None,16,"d5")
    output = layers.Conv2D(1,1,activation="sigmoid")(x)

    return keras.Model(inputs,output),encoder

model,encoder = build_model()

def compile_model(lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=combined_loss,
        metrics=[
            dice_coefficient,
            soft_iou,
            keras.metrics.BinaryAccuracy(name="pixel_accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

compile_model(1e-3)
model.summary()


In [ ]:
def training_callbacks(path,log,patience):
    return [keras.callbacks.ModelCheckpoint(str(path),monitor='val_loss',mode='min',save_best_only=True),
        keras.callbacks.EarlyStopping(monitor='val_loss',mode='min',patience=patience,restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=2,min_lr=5e-8),
        keras.callbacks.CSVLogger(str(OUTPUT_DIR/log))]
HEAD_MODEL, FINE_MODEL = OUTPUT_DIR/'head_best.keras', OUTPUT_DIR/'fine_best.keras'
head_history = model.fit(train_ds,validation_data=val_ds,epochs=HEAD_EPOCHS,
    callbacks=training_callbacks(HEAD_MODEL,'head_history.csv',5))
# Start fine-tuning from the best head weights, preserving encoder object identity.
model.load_weights(HEAD_MODEL)
encoder.trainable = True
fine_tune_start = int(len(encoder.layers)*.85)
for i,layer in enumerate(encoder.layers):
    layer.trainable = i >= fine_tune_start and not isinstance(layer,layers.BatchNormalization)
compile_model(5e-6)
fine_history = model.fit(train_ds,validation_data=val_ds,epochs=FINE_TUNE_EPOCHS,
    callbacks=training_callbacks(FINE_MODEL,'fine_tune_history.csv',4))
# Compare stages on validation only; never overwrite the better stage blindly.
stage_scores = []
for path in [HEAD_MODEL,FINE_MODEL]:
    model.load_weights(path)
    stage_scores.append((float(model.evaluate(val_ds,verbose=0,return_dict=True)['loss']),path))
selected_loss,selected_path = min(stage_scores,key=lambda x:x[0])
model.load_weights(selected_path)
BEST_MODEL = OUTPUT_DIR/'best_model.keras'
model.save(BEST_MODEL)
print('Selected',selected_path.name,'validation loss',selected_loss)


In [ ]:
def predict_dataframe(df):
    results=[]
    for row in df.itertuples():
        rgb,mask_native=load_pair_arrays(row.image_path,row.mask_path)
        image=preprocess_rgb(rgb)
        mask=tf.image.resize(mask_native[...,None],[IMAGE_SIZE,IMAGE_SIZE],method='nearest')[...,0].numpy()
        prob=np.asarray(model(image[None,...],training=False))[0,...,0]
        results.append(dict(stem=row.stem,coverage_bin=row.coverage_bin,image=image.numpy(),mask=mask,probability=prob))
    return results
val_predictions=predict_dataframe(val_df)
# Test predictions are deliberately deferred until threshold selection is complete.


In [ ]:
def image_metrics(true_mask,probability,threshold):
    true_mask = true_mask >= 0.5
    pred = probability >= threshold

    inter = np.logical_and(true_mask,pred).sum()
    union = np.logical_or(true_mask,pred).sum()
    tp = inter
    fp = np.logical_and(~true_mask,pred).sum()
    fn = np.logical_and(true_mask,~pred).sum()

    if true_mask.sum()==0 and pred.sum()==0:
        iou,dice = 1.0,1.0
    else:
        iou = inter/(union+1e-8)
        dice = 2*inter/(true_mask.sum()+pred.sum()+1e-8)

    true_cov = true_mask.mean()*100
    pred_cov = pred.mean()*100

    return {
        "iou":float(iou),
        "dice":float(dice),
        "precision":float(tp/(tp+fp+1e-8)),
        "recall":float(tp/(tp+fn+1e-8)),
        "true_coverage":float(true_cov),
        "predicted_coverage":float(pred_cov),
        "coverage_error":float(pred_cov-true_cov),
        "coverage_abs_error":float(abs(pred_cov-true_cov)),
    }


In [ ]:
def threshold_sweep(predictions):
    if not predictions: raise ValueError('No validation images')
    rows=[]
    for threshold in np.round(np.arange(.05,.951,.05),2):
        metrics=[image_metrics(x['mask'],x['probability'],threshold) for x in predictions]
        positive=[v for v in metrics if v['true_coverage']>0]
        negative=[v for v in metrics if v['true_coverage']==0]
        rows.append(dict(threshold=float(threshold),mean_iou=float(np.mean([v['iou'] for v in metrics])),
            positive_mean_iou=float(np.mean([v['iou'] for v in positive])) if positive else np.nan,
            coverage_mae=float(np.mean([v['coverage_abs_error'] for v in metrics])),
            negative_mean_false_coverage=float(np.mean([v['predicted_coverage'] for v in negative])) if negative else np.nan))
    table=pd.DataFrame(rows)
    if table.positive_mean_iou.isna().all(): raise ValueError('Validation needs positive examples')
    candidates=table[table.positive_mean_iou>=table.positive_mean_iou.max()-.02]
    chosen=candidates.sort_values(['coverage_mae','positive_mean_iou'],ascending=[True,False]).iloc[0]
    return table,float(chosen.threshold)
threshold_df,BEST_THRESHOLD=threshold_sweep(val_predictions)
display(threshold_df); print('Validation-selected threshold',BEST_THRESHOLD)
threshold_df.to_csv(OUTPUT_DIR/'threshold_sweep.csv',index=False)
test_predictions=predict_dataframe(test_df)


In [ ]:
test_rows = []

for item in test_predictions:
    test_rows.append({
        "stem":item["stem"],
        "coverage_bin":item["coverage_bin"],
        "group_id":test_df.set_index("stem").loc[item["stem"],"group_id"],
        **image_metrics(
            item["mask"],
            item["probability"],
            BEST_THRESHOLD,
        ),
    })

test_results = pd.DataFrame(test_rows)
display(test_results)

summary = {
    "test_images":int(len(test_results)),
    "threshold":BEST_THRESHOLD,
    "mean_iou":float(test_results["iou"].mean()),
    "median_iou":float(test_results["iou"].median()),
    "mean_dice":float(test_results["dice"].mean()),
    "mean_precision":float(test_results["precision"].mean()),
    "mean_recall":float(test_results["recall"].mean()),
    "coverage_mae_percentage_points":
        float(test_results["coverage_abs_error"].mean()),
    "coverage_bias_percentage_points":
        float(test_results["coverage_error"].mean()),
}

print(json.dumps(summary,indent=2))

test_results.to_csv(OUTPUT_DIR/"test_metrics.csv",index=False)
with open(OUTPUT_DIR/"test_summary.json","w") as f:
    json.dump(summary,f,indent=2)


In [ ]:
def bootstrap_ci(values,samples=5000,groups=None):
    rng=np.random.default_rng(SEED)
    values=np.asarray(values,dtype=float)
    if len(values)==0: return None
    groups=np.asarray(groups) if groups is not None else np.arange(len(values))
    if len(groups)!=len(values): raise ValueError('Bootstrap group length mismatch')
    unique=np.unique(groups)
    estimates=[]
    for _ in range(samples):
        sampled=rng.choice(unique,size=len(unique),replace=True)
        estimates.append(float(np.concatenate([values[groups==g] for g in sampled]).mean()))
    return np.percentile(estimates,[2.5,97.5]).tolist()
cis={
    'mean_iou_95_ci':bootstrap_ci(test_results.iou,groups=test_results.group_id),
    'mean_dice_95_ci':bootstrap_ci(test_results.dice,groups=test_results.group_id),
    'coverage_mae_95_ci':bootstrap_ci(test_results.coverage_abs_error,groups=test_results.group_id),
    'method':'percentile bootstrap resampling source/duplicate groups',
    'independent_test_groups':int(test_results.group_id.nunique()),
}
print(json.dumps(cis,indent=2))
(OUTPUT_DIR/'confidence_intervals.json').write_text(json.dumps(cis,indent=2))


In [ ]:
category_results = (
    test_results.groupby("coverage_bin")
    .agg(
        images=("stem","count"),
        mean_iou=("iou","mean"),
        mean_dice=("dice","mean"),
        coverage_mae=("coverage_abs_error","mean"),
        coverage_bias=("coverage_error","mean"),
    )
    .reset_index()
)

display(category_results)
category_results.to_csv(
    OUTPUT_DIR/"test_metrics_by_category.csv",
    index=False,
)


In [ ]:
positive_results = test_results[
    test_results["true_coverage"] > 0
].copy()

negative_results = test_results[
    test_results["true_coverage"] == 0
].copy()

positive_mean_iou = positive_results["iou"].mean()
positive_median_iou = positive_results["iou"].median()

negative_mean_false_coverage = (
    negative_results["predicted_coverage"].mean()
    if len(negative_results) > 0
    else None
)

negative_below_2_percent_rate = (
    (negative_results["predicted_coverage"] < 2.0).mean()
    if len(negative_results) > 0
    else None
)

separated_summary = {
    "positive_test_images": int(len(positive_results)),
    "positive_mean_iou": float(positive_mean_iou),
    "positive_median_iou": float(positive_median_iou),
    "negative_test_images": int(len(negative_results)),
    "negative_mean_false_coverage_percent": (float(negative_mean_false_coverage) if negative_mean_false_coverage is not None else None),
    "negative_below_2_percent_rate": (
        float(negative_below_2_percent_rate)
        if negative_below_2_percent_rate is not None
        else None
    ),
    "overall_coverage_mae": float(
        test_results["coverage_abs_error"].mean()
    ),
}

print(json.dumps(separated_summary, indent=2))
(OUTPUT_DIR/'separated_summary.json').write_text(json.dumps(separated_summary,indent=2))


In [ ]:
def show_prediction(item):
    m = image_metrics(
        item["mask"],
        item["probability"],
        BEST_THRESHOLD,
    )
    pred = item["probability"] >= BEST_THRESHOLD

    plt.figure(figsize=(16,4))
    plt.subplot(1,4,1)
    plt.imshow(item["image"])
    plt.title(item["stem"])
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(item["mask"],cmap="gray")
    plt.title(f"True: {m['true_coverage']:.1f}%")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(item["probability"],cmap="gray",vmin=0,vmax=1)
    plt.title("Probability")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(pred,cmap="gray")
    plt.title(
        f"Pred: {m['predicted_coverage']:.1f}%\n"
        f"IoU: {m['iou']:.3f}"
    )
    plt.axis("off")
    plt.show()

worst = test_results.sort_values(
    ["iou","coverage_abs_error"],
    ascending=[True,False],
)

display(worst.head(12))

for stem in worst.head(min(12,len(worst)))["stem"]:
    item = next(x for x in test_predictions if x["stem"]==stem)
    show_prediction(item)


In [ ]:
FINAL_MODEL=OUTPUT_DIR/'lichen_coverage_model_v3.keras'
model.save(FINAL_MODEL)
metadata=dict(version=3,image_size=IMAGE_SIZE,threshold=BEST_THRESHOLD,
    architecture='MobileNetV2 U-Net',input_contract='RGB float32 [0,1]; internal 2*x-1',
    image_orientation='EXIF transpose',resize='bilinear antialias=True, square warp',
    mask_resize='nearest',mask_background=MASK_BACKGROUND,mask_foreground=MASK_FOREGROUND,
    mask_coordinates=MASK_COORDINATES,coverage_grid='320x320; threshold probability on this grid',
    coverage_denominator='entire photograph, including background',
    training_images=len(train_df),validation_images=len(val_df),test_images=len(test_df),
    source_groups_supplied=source_groups is not None,test_summary=summary,confidence_intervals=cis,
    selected_stage=selected_path.name,model_sha256=hashlib.sha256(FINAL_MODEL.read_bytes()).hexdigest(),
    status='requires independent labeled field validation')
(OUTPUT_DIR/'model_metadata.json').write_text(json.dumps(metadata,indent=2))
# Save/reload parity is part of the workflow, not an assumed property.
probe=read_image(val_df.iloc[0].image_path)[None,...]
reloaded=keras.models.load_model(FINAL_MODEL,compile=False)
np.testing.assert_allclose(model(probe,training=False),reloaded(probe,training=False),atol=1e-6,rtol=1e-5)
print('Saved',FINAL_MODEL,'; save/reload parity passed')


## Field inference and independent evaluation

The deployed wrapper must use this same preprocessing, the metadata beside the checkpoint, and the same coverage grid. Never normalize twice or substitute a threshold from another run. Square resize is retained consistently; it distorts aspect ratio and can erase fine patches. Letterboxing/tiling may be useful experiments, but they require their own validation and recalibration. Probabilities stay float32; resize probabilities bilinearly only for display and binary masks with nearest neighbor. Never resize a binary mask bilinearly and then count nonzero pixels.

Coverage here means fraction of the **whole photo**, not fraction of bark/rock surface. To report substrate coverage, add and validate a substrate ROI. Low predicted coverage is not proof of absence. Mean `max(p,1-p)` is pixel certainty, not calibrated accuracy or an out-of-domain detector.


In [ ]:
def load_deployment(model_path):
    model_path=Path(model_path)
    meta=json.loads((model_path.parent/'model_metadata.json').read_text())
    if hashlib.sha256(model_path.read_bytes()).hexdigest()!=meta['model_sha256']:
        raise ValueError('Checkpoint and metadata do not match')
    deployed=keras.models.load_model(model_path,compile=False)
    if deployed.input_shape[1:]!=(meta['image_size'],meta['image_size'],3):
        raise ValueError('Unexpected model input shape')
    return deployed,meta

def infer_field(path,deployed,meta):
    if meta['image_size']!=IMAGE_SIZE: raise ValueError('Preprocessor size mismatch')
    x=read_image(path)
    prob=np.asarray(deployed(x[None,...],training=False))[0,...,0].astype(np.float32)
    if prob.shape!=(IMAGE_SIZE,IMAGE_SIZE) or not np.isfinite(prob).all() or prob.min()<0 or prob.max()>1:
        raise ValueError('Invalid probability map')
    mask=prob>=float(meta['threshold'])
    return dict(rgb=x.numpy(),prob=prob,mask=mask,coverage=float(mask.mean()*100),
        pixel_certainty=float(np.maximum(prob,1-prob).mean()),
        warning='Low prediction does not establish absence; field reliability requires labeled validation')

deployed,deploy_meta=load_deployment(FINAL_MODEL)
from google.colab import files
uploaded=files.upload()
FIELD_DIR=OUTPUT_DIR/'field_predictions'; FIELD_DIR.mkdir(exist_ok=True)
field_rows=[]
for name,content in uploaded.items():
    if Path(name).suffix.lower() not in EXTENSIONS: continue
    path=FIELD_DIR/Path(name).name; path.write_bytes(content)
    result=infer_field(path,deployed,deploy_meta)
    np.save(FIELD_DIR/(path.name+'.probability.npy'),result['prob'])
    Image.fromarray(result['mask'].astype(np.uint8)*255).save(FIELD_DIR/(path.name+'.mask.png'))
    field_rows.append(dict(filename=path.name,coverage_percent=result['coverage'],threshold=deploy_meta['threshold']))
    fig,ax=plt.subplots(1,3,figsize=(12,4))
    ax[0].imshow(result['rgb']); ax[1].imshow(result['prob'],vmin=0,vmax=1)
    ax[2].imshow(result['mask'],cmap='gray'); fig.suptitle(f'{path.name}: {result["coverage"]:.2f}% (unvalidated field prediction)')
    plt.show()
pd.DataFrame(field_rows).to_csv(FIELD_DIR/'predictions.csv',index=False)


In [ ]:
# Optional independent labeled field holdout: columns stem,image_path,mask_path,group_id.
# Paths must be readable in Colab. Use the same mask contract. Do not retrain or tune
# thresholds after looking at this holdout; collect a new holdout for the next iteration.
FIELD_MANIFEST=PROJECT_DIR/'field_holdout.csv'
if FIELD_MANIFEST.exists():
    field_df=pd.read_csv(FIELD_MANIFEST,dtype={'stem':str,'group_id':str})
    if not {'stem','image_path','mask_path','group_id'}.issubset(field_df.columns):
        raise ValueError('Invalid field manifest')
    if field_df.empty or field_df.stem.duplicated().any() or field_df.group_id.isna().any():
        raise ValueError('Empty field holdout, duplicate stems or missing group IDs')
    if source_groups is not None and set(field_df.group_id)&set(source_groups.group_id):
        raise ValueError('Field holdout shares source groups with development data')
    field_df['coverage_bin']='field'
    for r in field_df.itertuples():
        rgb,_=load_pair_arrays(r.image_path,r.mask_path)
        sha=hashlib.sha256(str(rgb.shape).encode()+rgb.tobytes()).hexdigest()
        h=dhash(rgb)
        if sha in set(pairs.pixel_sha256) or any((int(h,16)^int(v,16)).bit_count()<=4 for v in pairs.dhash):
            raise ValueError(f'Field holdout duplicates/resembles development data: {r.stem}')
    field_predictions=predict_dataframe(field_df)
    field_results=pd.DataFrame([dict(stem=x['stem'],**image_metrics(x['mask'],x['probability'],BEST_THRESHOLD)) for x in field_predictions])
    field_results.to_csv(OUTPUT_DIR/'field_holdout_metrics.csv',index=False)
    display(field_results)
    print('Field coverage MAE (percentage points):',field_results.coverage_abs_error.mean())
else:
    print('No labeled field holdout supplied. Field accuracy and improvement remain unverified.')


Do not declare readiness from the 21-image split alone. The original validation contained no very-low-coverage images and just one negative; test had one of each. Report counts and performance by growth form, substrate, camera/distance and site on independent field data, including hard negatives. Review systematic false negatives before claiming absence. Broader augmentation is a starting experiment; representative, consistently annotated photographs are needed to establish improvement.

References: [Keras MobileNetV2 input contract](https://keras.io/api/applications/mobilenet/mobilenet_models/), [Keras transfer learning](https://keras.io/guides/transfer_learning/), [MobileNetV2 source graph construction](https://github.com/keras-team/keras/blob/master/keras/src/applications/mobilenet_v2.py).


In [ ]:
# One export includes both checkpoints and all new evaluation artifacts, without
# overwriting any older training_output directory.
import shutil
bundle=shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR)
files.download(bundle)


# Historical appendix — original sources and saved outputs

Everything below is preserved evidence from the uploaded notebook, **not execution of the revised workflow**. Original source is rendered as text so Run All cannot accidentally retrain or overwrite the old model. Output-only cells preserve every original MIME bundle, plot, warning and execution count. The unmodified original notebook is also provided separately.


### Original cell 0

# EcoSignal Final Coverage Retraining v2

This Colab notebook retrains the lichen coverage model from scratch using the completed dataset.

Changes from the previous notebook:
- 20% untouched test set
- coverage-stratified splitting, including pure negatives
- 320×320 resolution
- lighter fine-tuning to reduce overfitting
- threshold selected to balance IoU and coverage MAE
- bootstrap confidence intervals


### Original cell 1

```python
!rm -rf /usr/local/lib/python3.13/dist-packages/~ensorflow*

```


### Original cell 2

```python
!pip -q install --force-reinstall -U tensorflow scikit-learn pillow pandas matplotlib

import json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
```


In [ ]:
# Historical output from original cell 2. No code is executed here.


### Original cell 3

```python
from google.colab import drive
drive.mount("/content/drive")

```


In [ ]:
# Historical output from original cell 3. No code is executed here.


### Original cell 4

Please upload your zip files (e.g., `images.zip` and `masks.zip`) to your Colab environment. Once uploaded, run the following cells to extract them and configure the data paths.


### Original cell 5

```python
import zipfile
import os

# Define the paths for your zip files and extraction directories
# Replace 'images.zip' and 'masks.zip' with the actual names of your uploaded files
IMAGES_ZIP_PATH = 'Images.zip'
MASKS_ZIP_PATH = 'Masks.zip'

# Define the directories where the zip files will be extracted
# These will be temporary directories within the Colab environment
EXTRACTED_IMAGE_DIR = Path('/tmp/extracted_images')
EXTRACTED_MASK_DIR = Path('/tmp/extracted_masks')

# Create extraction directories if they don't exist
EXTRACTED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_MASK_DIR.mkdir(parents=True, exist_ok=True)

# Extract images zip file
if os.path.exists(IMAGES_ZIP_PATH):
    with zipfile.ZipFile(IMAGES_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_IMAGE_DIR)
    print(f"Extracted {IMAGES_ZIP_PATH} to {EXTRACTED_IMAGE_DIR}")
else:
    print(f"Warning: {IMAGES_ZIP_PATH} not found. Please ensure you have uploaded it.")

# Extract masks zip file
if os.path.exists(MASKS_ZIP_PATH):
    with zipfile.ZipFile(MASKS_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_MASK_DIR)
    print(f"Extracted {MASKS_ZIP_PATH} to {EXTRACTED_MASK_DIR}")
else:
    print(f"Warning: {MASKS_ZIP_PATH} not found. Please ensure you have uploaded it.")

```


In [ ]:
# Historical output from original cell 5. No code is executed here.


### Original cell 6

Now, let's update the `IMAGE_DIR` and `MASK_DIR` to use the extracted datasets. This will override the previous paths pointing to Google Drive.


### Original cell 7

```python
# Update IMAGE_DIR and MASK_DIR to use the extracted files
IMAGE_DIR = EXTRACTED_IMAGE_DIR
MASK_DIR = EXTRACTED_MASK_DIR

print(f"IMAGE_DIR set to: {IMAGE_DIR}")
print(f"MASK_DIR set to: {MASK_DIR}")

```


In [ ]:
# Historical output from original cell 7. No code is executed here.


### Original cell 8

```python
PROJECT_DIR = Path("/content/drive/MyDrive/EcoSignalCoverage")
IMAGE_DIR = PROJECT_DIR / "images"
MASK_DIR = PROJECT_DIR / "masks"
OUTPUT_DIR = PROJECT_DIR / "training_output" / "final_retraining_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 320
BATCH_SIZE = 4
HEAD_EPOCHS = 18
FINE_TUNE_EPOCHS = 10
VAL_FRACTION = 0.15
TEST_FRACTION = 0.20
SEED = 42

EXTENSIONS = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff", ".avif"}

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

```


### Original cell 9

```python
IMAGE_DIR = EXTRACTED_IMAGE_DIR / "Images"
MASK_DIR = EXTRACTED_MASK_DIR / "Masks"

def files_by_stem(folder):
    return {
        p.stem: p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    }

images = files_by_stem(IMAGE_DIR)
mass = files_by_stem(MASK_DIR)
stems = sorted(set(images) & set(mass))

pairs = pd.DataFrame({
    "stem": stems,
    "image_path": [str(images[s]) for s in stems],
    "mask_path": [str(mass[s]) for s in stems],
})

print("Images:", len(images))
print("Masks:", len(mass))
print("Matched:", len(pairs))
print("Missing masks:", sorted(set(images)-set(mass))[:20])
print("Missing images:", sorted(set(mass)-set(images))[:20])

if len(pairs) < 40:
    raise ValueError("Too few matched image-mask pairs.")
```


In [ ]:
# Historical output from original cell 9. No code is executed here.


### Original cell 10

```python
def mask_coverage(mask_path):
    mask = np.asarray(Image.open(mask_path).convert("L")) > 0
    return float(mask.mean() * 100)

pairs["coverage_percent"] = pairs["mask_path"].map(mask_coverage)

def coverage_bin(value):
    if value == 0: return "negative"
    if value < 10: return "very_low"
    if value < 30: return "low"
    if value < 60: return "medium"
    return "high"

pairs["coverage_bin"] = pairs["coverage_percent"].map(coverage_bin)

display(pairs["coverage_bin"].value_counts())
display(pairs["coverage_percent"].describe())

plt.figure(figsize=(8,4))
plt.hist(pairs["coverage_percent"], bins=20)
plt.xlabel("Ground-truth coverage (%)")
plt.ylabel("Images")
plt.show()

```


In [ ]:
# Historical output from original cell 10. No code is executed here.


### Original cell 11

```python
def safe_labels(df):
    labels = df["coverage_bin"].copy()
    counts = labels.value_counts()

    # Identify categories with less than 3 samples to group into 'other'
    rare_initial = counts[counts < 3].index

    # Apply initial grouping
    processed_labels = labels.where(~labels.isin(rare_initial), "other")

    # Check the counts of the processed labels
    processed_counts = processed_labels.value_counts()

    # Identify if any class in the processed_labels still has less than 2 members
    problematic_classes = processed_counts[processed_counts < 2].index

    if len(problematic_classes) > 0:
        # If 'other' is a problematic class and has only 1 member, reassign it
        if "other" in problematic_classes and processed_counts["other"] == 1:
            # Find the largest non-'other' class to assign the singleton to
            largest_class = processed_counts[processed_counts.index != "other"].idxmax()

            # Find the index of the single 'other' sample
            other_index = processed_labels[processed_labels == "other"].index[0]

            # Reassign this single 'other' sample to the largest class
            processed_labels.loc[other_index] = largest_class

            print(f"Warning: 'other' class had only 1 member. Reassigned it to '{largest_class}' for stratification.")
        else:
            # If other problematic classes exist (e.g., another original class with 1 member after grouping),
            # or if 'other' has more than 1 but still less than 2 members, this case would need further review.
            # For simplicity, we assume 'other' is the only problematic class as per the error.
            pass # If other cases, current logic might not cover all edge cases.

    return processed_labels

labels = safe_labels(pairs)

train_df, temp_df = train_test_split(
    pairs,
    test_size=VAL_FRACTION + TEST_FRACTION,
    random_state=SEED,
    shuffle=True,
    stratify=labels,
)

temp_labels = safe_labels(temp_df)
relative_test = TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test,
    random_state=SEED,
    shuffle=True,
    stratify=temp_labels,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

for name, df in [("Train",train_df),("Validation",val_df),("Test",test_df)]:
    print(name)
    display(df["coverage_bin"].value_counts())

with open(OUTPUT_DIR / "split_manifest.json","w") as f:
    json.dump({
        "train": train_df["stem"].tolist(),
        "validation": val_df["stem"].tolist(),
        "test": test_df["stem"].tolist(),
    }, f, indent=2)

```


In [ ]:
# Historical output from original cell 11. No code is executed here.


### Original cell 12

```python
AUTOTUNE = tf.data.AUTOTUNE

def read_image(path):
    data = tf.io.read_file(path)
    image = tf.io.decode_image(data, channels=3, expand_animations=False)
    image.set_shape([None,None,3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    return tf.image.resize(image, [IMAGE_SIZE,IMAGE_SIZE], method="bilinear")

def read_mask(path):
    data = tf.io.read_file(path)
    mask = tf.io.decode_image(data, channels=1, expand_animations=False)
    mask.set_shape([None,None,1])
    mask = tf.cast(mask > 0, tf.float32)
    mask = tf.image.resize(mask, [IMAGE_SIZE,IMAGE_SIZE], method="nearest")
    return tf.cast(mask >= 0.5, tf.float32)

def load_pair(image_path, mask_path):
    return read_image(image_path), read_mask(mask_path)

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform([],0,4,dtype=tf.int32)
    image = tf.image.rot90(image,k)
    mask = tf.image.rot90(mask,k)

    image = tf.image.random_brightness(image,0.08)
    image = tf.image.random_contrast(image,0.88,1.12)
    image = tf.image.random_saturation(image,0.88,1.12)
    image += tf.random.normal(tf.shape(image),stddev=0.01)
    image = tf.clip_by_value(image,0,1)
    return image,mask

def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["image_path"].values,df["mask_path"].values)
    )
    ds = ds.map(load_pair,num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
        ds = ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df,True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

```


### Original cell 13

```python
SMOOTH = 1e-6

def dice_coefficient(y_true,y_pred):
    yt = tf.reshape(tf.cast(y_true,tf.float32),[tf.shape(y_true)[0],-1])
    yp = tf.reshape(tf.cast(y_pred,tf.float32),[tf.shape(y_pred)[0],-1])
    inter = tf.reduce_sum(yt*yp,axis=1)
    denom = tf.reduce_sum(yt,axis=1)+tf.reduce_sum(yp,axis=1)
    return tf.reduce_mean((2*inter+SMOOTH)/(denom+SMOOTH))

def soft_iou(y_true,y_pred):
    yt = tf.reshape(tf.cast(y_true,tf.float32),[tf.shape(y_true)[0],-1])
    yp = tf.reshape(tf.cast(y_pred,tf.float32),[tf.shape(y_pred)[0],-1])
    inter = tf.reduce_sum(yt*yp,axis=1)
    union = tf.reduce_sum(yt,axis=1)+tf.reduce_sum(yp,axis=1)-inter
    return tf.reduce_mean((inter+SMOOTH)/(union+SMOOTH))

bce = keras.losses.BinaryCrossentropy()

def combined_loss(y_true,y_pred):
    return 0.5*bce(y_true,y_pred)+0.5*(1-dice_coefficient(y_true,y_pred))

```


### Original cell 14

```python
def decoder_block(x,skip,filters,name):
    x = layers.Conv2DTranspose(filters,3,strides=2,padding="same",
                               use_bias=False,name=name+"_up")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    if skip is not None:
        x = layers.Concatenate()([x,skip])
    for _ in range(2):
        x = layers.Conv2D(filters,3,padding="same",use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x

def build_model():
    inputs = keras.Input((IMAGE_SIZE,IMAGE_SIZE,3))
    x0 = layers.Rescaling(2.0,offset=-1.0)(inputs)

    base = keras.applications.MobileNetV2(
        input_tensor=x0, include_top=False, weights="imagenet"
    )

    skip_names = [
        "block_1_expand_relu",
        "block_3_expand_relu",
        "block_6_expand_relu",
        "block_13_expand_relu",
    ]

    outputs = [base.get_layer(n).output for n in skip_names]
    outputs.append(base.get_layer("block_16_project").output)
    encoder = keras.Model(base.input,outputs,name="encoder")
    encoder.trainable = False

    s1,s2,s3,s4,x = encoder(inputs)
    x = decoder_block(x,s4,256,"d1")
    x = decoder_block(x,s3,128,"d2")
    x = decoder_block(x,s2,64,"d3")
    x = decoder_block(x,s1,32,"d4")
    x = decoder_block(x,None,16,"d5")
    output = layers.Conv2D(1,1,activation="sigmoid")(x)

    return keras.Model(inputs,output),encoder

model,encoder = build_model()

def compile_model(lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=combined_loss,
        metrics=[
            dice_coefficient,
            soft_iou,
            keras.metrics.BinaryAccuracy(name="pixel_accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

compile_model(1e-3)
model.summary()

```


In [ ]:
# Historical output from original cell 14. No code is executed here.


### Original cell 15

```python
import io
from PIL import Image

# Helper function for read_image to be used with tf.py_function
def _read_image_py_logic(path_bytes_tensor):
    path_str = path_bytes_tensor.numpy().decode('utf-8')

    if any(path_str.lower().endswith(ext) for ext in [".tif", ".tiff", ".avif"]):
        img = Image.open(path_str).convert("RGB") # Ensure 3 channels
        img_array = np.array(img)
        image = tf.convert_to_tensor(img_array, dtype=tf.float32)
        image = image / 255.0 # Normalize pixel values to [0, 1]
    else:
        # For other formats, use Python's file open and then TF's decoder
        with open(path_str, 'rb') as f:
            image_bytes = f.read()
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image = tf.image.convert_image_dtype(image, tf.float32)

    # Set a known shape for the image tensor. The height/width can be dynamic.
    image.set_shape([None, None, 3])
    return image

# Redefine read_image to use tf.py_function
def read_image(path):
    image = tf.py_function(
        func=_read_image_py_logic,
        inp=[path],
        Tout=tf.float32
    )
    # The shape needs to be set explicitly after tf.py_function call for graph compatibility
    image.set_shape([None, None, 3])
    return tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method="bilinear")

# Helper function for read_mask to be used with tf.py_function
def _read_mask_py_logic(path_bytes_tensor):
    path_str = path_bytes_tensor.numpy().decode('utf-8')

    if any(path_str.lower().endswith(ext) for ext in [".tif", ".tiff", ".avif"]):
        mask_img = Image.open(path_str).convert("L") # Convert to grayscale
        mask_array = np.array(mask_img)
        mask = tf.convert_to_tensor(mask_array, dtype=tf.float32)
        mask = tf.expand_dims(mask, -1) # Add channel dimension
        mask = tf.cast(mask > 0, tf.float32) # Ensure binary mask
    else:
        # For other formats, use Python's file open and then TF's decoder
        with open(path_str, 'rb') as f:
            mask_bytes = f.read()
        mask = tf.io.decode_image(mask_bytes, channels=1, expand_animations=False)
        mask = tf.cast(mask > 0, tf.float32)

    # Set a known shape for the mask tensor
    mask.set_shape([None, None, 1])
    return mask

# Redefine read_mask to use tf.py_function
def read_mask(path):
    mask = tf.py_function(
        func=_read_mask_py_logic,
        inp=[path],
        Tout=tf.float32
    )
    # The shape needs to be set explicitly after tf.py_function call for graph compatibility
    mask.set_shape([None, None, 1])
    mask = tf.image.resize(mask, [IMAGE_SIZE, IMAGE_SIZE], method="nearest")
    return tf.cast(mask >= 0.5, tf.float32)

# Redefine load_pair as it depends on the now-overridden read_image and read_mask
def load_pair(image_path, mask_path):
    return read_image(image_path), read_mask(mask_path)

# Redefine make_dataset as it depends on the now-overridden load_pair
def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["image_path"].values,df["mask_path"].values)
    )
    ds = ds.map(load_pair,num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
        ds = ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Recreate the datasets with the fixed functions
train_ds = make_dataset(train_df,True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df) # Re-create test_ds for consistency, though not directly used here.

BEST_MODEL = OUTPUT_DIR / "best_model.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),monitor="val_soft_iou",
        mode="max",save_best_only=True,verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_soft_iou",mode="max",
        patience=5,restore_best_weights=True,verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",factor=0.5,
        patience=2,min_lr=1e-6,verbose=1
    ),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR/"head_history.csv")),
]

head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks,
)

```


In [ ]:
# Historical output from original cell 15. No code is executed here.


### Original cell 16

```python
encoder.trainable = True
fine_tune_start = int(len(encoder.layers)*0.85)

for layer in encoder.layers[:fine_tune_start]:
    layer.trainable = False

for layer in encoder.layers:
    if isinstance(layer,layers.BatchNormalization):
        layer.trainable = False

compile_model(5e-6)

fine_callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),monitor="val_soft_iou",
        mode="max",save_best_only=True,verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_soft_iou",mode="max",
        patience=4,restore_best_weights=True,verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",factor=0.5,
        patience=2,min_lr=5e-8,verbose=1
    ),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR/"fine_tune_history.csv")),
]

fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_callbacks,
)

```


In [ ]:
# Historical output from original cell 16. No code is executed here.


### Original cell 17

```python
custom_objects = {
    "combined_loss":combined_loss,
    "dice_coefficient":dice_coefficient,
    "soft_iou":soft_iou,
}

model = keras.models.load_model(
    BEST_MODEL,
    custom_objects=custom_objects,
)

```


### Original cell 18

```python
def predict_dataframe(df):
    results = []
    for row in df.itertuples():
        image = read_image(row.image_path)
        mask = read_mask(row.mask_path)[...,0]
        prob = model.predict(image[None,...],verbose=0)[0,...,0]
        results.append({
            "stem":row.stem,
            "coverage_bin":row.coverage_bin,
            "image":image.numpy(),
            "mask":mask.numpy(),
            "probability":prob,
        })
    return results

val_predictions = predict_dataframe(val_df)
test_predictions = predict_dataframe(test_df)

```


### Original cell 19

```python
def image_metrics(true_mask,probability,threshold):
    true_mask = true_mask >= 0.5
    pred = probability >= threshold

    inter = np.logical_and(true_mask,pred).sum()
    union = np.logical_or(true_mask,pred).sum()
    tp = inter
    fp = np.logical_and(~true_mask,pred).sum()
    fn = np.logical_and(true_mask,~pred).sum()

    if true_mask.sum()==0 and pred.sum()==0:
        iou,dice = 1.0,1.0
    else:
        iou = inter/(union+1e-8)
        dice = 2*inter/(true_mask.sum()+pred.sum()+1e-8)

    true_cov = true_mask.mean()*100
    pred_cov = pred.mean()*100

    return {
        "iou":float(iou),
        "dice":float(dice),
        "precision":float(tp/(tp+fp+1e-8)),
        "recall":float(tp/(tp+fn+1e-8)),
        "true_coverage":float(true_cov),
        "predicted_coverage":float(pred_cov),
        "coverage_error":float(pred_cov-true_cov),
        "coverage_abs_error":float(abs(pred_cov-true_cov)),
    }

```


### Original cell 20

```python
rows = []

for threshold in np.arange(0.20,0.86,0.05):
    values = [
        image_metrics(x["mask"],x["probability"],float(threshold))
        for x in val_predictions
    ]
    rows.append({
        "threshold":float(threshold),
        "mean_iou":np.mean([v["iou"] for v in values]),
        "mean_dice":np.mean([v["dice"] for v in values]),
        "coverage_mae":np.mean([v["coverage_abs_error"] for v in values]),
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)

best_iou = threshold_df["mean_iou"].max()
candidates = threshold_df[
    threshold_df["mean_iou"] >= best_iou - 0.02
]

best_row = candidates.sort_values(
    ["coverage_mae","mean_iou"],
    ascending=[True,False],
).iloc[0]

BEST_THRESHOLD = float(best_row["threshold"])
print("Selected threshold:",BEST_THRESHOLD)

threshold_df.to_csv(OUTPUT_DIR/"threshold_sweep.csv",index=False)

```


In [ ]:
# Historical output from original cell 20. No code is executed here.


### Original cell 21

```python
test_rows = []

for item in test_predictions:
    test_rows.append({
        "stem":item["stem"],
        "coverage_bin":item["coverage_bin"],
        **image_metrics(
            item["mask"],
            item["probability"],
            BEST_THRESHOLD,
        ),
    })

test_results = pd.DataFrame(test_rows)
display(test_results)

summary = {
    "test_images":int(len(test_results)),
    "threshold":BEST_THRESHOLD,
    "mean_iou":float(test_results["iou"].mean()),
    "median_iou":float(test_results["iou"].median()),
    "mean_dice":float(test_results["dice"].mean()),
    "mean_precision":float(test_results["precision"].mean()),
    "mean_recall":float(test_results["recall"].mean()),
    "coverage_mae_percentage_points":
        float(test_results["coverage_abs_error"].mean()),
    "coverage_bias_percentage_points":
        float(test_results["coverage_error"].mean()),
}

print(json.dumps(summary,indent=2))

test_results.to_csv(OUTPUT_DIR/"test_metrics.csv",index=False)
with open(OUTPUT_DIR/"test_summary.json","w") as f:
    json.dump(summary,f,indent=2)

```


In [ ]:
# Historical output from original cell 21. No code is executed here.


### Original cell 22

```python
def bootstrap_ci(values,samples=5000):
    rng = np.random.default_rng(SEED)
    values = np.asarray(values)
    estimates = [
        np.mean(rng.choice(values,size=len(values),replace=True))
        for _ in range(samples)
    ]
    return [float(x) for x in np.percentile(estimates,[2.5,97.5])]

cis = {
    "mean_iou_95_ci":bootstrap_ci(test_results["iou"]),
    "mean_dice_95_ci":bootstrap_ci(test_results["dice"]),
    "coverage_mae_95_ci":
        bootstrap_ci(test_results["coverage_abs_error"]),
}

print(json.dumps(cis,indent=2))

with open(OUTPUT_DIR/"confidence_intervals.json","w") as f:
    json.dump(cis,f,indent=2)

```


In [ ]:
# Historical output from original cell 22. No code is executed here.


### Original cell 23

```python
category_results = (
    test_results.groupby("coverage_bin")
    .agg(
        images=("stem","count"),
        mean_iou=("iou","mean"),
        mean_dice=("dice","mean"),
        coverage_mae=("coverage_abs_error","mean"),
        coverage_bias=("coverage_error","mean"),
    )
    .reset_index()
)

display(category_results)
category_results.to_csv(
    OUTPUT_DIR/"test_metrics_by_category.csv",
    index=False,
)

```


In [ ]:
# Historical output from original cell 23. No code is executed here.


### Original cell 24

```python
positive_results = test_results[
    test_results["true_coverage"] > 0
].copy()

negative_results = test_results[
    test_results["true_coverage"] == 0
].copy()

positive_mean_iou = positive_results["iou"].mean()
positive_median_iou = positive_results["iou"].median()

negative_mean_false_coverage = (
    negative_results["predicted_coverage"].mean()
    if len(negative_results) > 0
    else 0.0
)

negative_below_2_percent_rate = (
    (negative_results["predicted_coverage"] < 2.0).mean()
    if len(negative_results) > 0
    else None
)

separated_summary = {
    "positive_test_images": int(len(positive_results)),
    "positive_mean_iou": float(positive_mean_iou),
    "positive_median_iou": float(positive_median_iou),
    "negative_test_images": int(len(negative_results)),
    "negative_mean_false_coverage_percent": float(
        negative_mean_false_coverage
    ),
    "negative_below_2_percent_rate": (
        float(negative_below_2_percent_rate)
        if negative_below_2_percent_rate is not None
        else None
    ),
    "overall_coverage_mae": float(
        test_results["coverage_abs_error"].mean()
    ),
}

print(json.dumps(separated_summary, indent=2))
```


In [ ]:
# Historical output from original cell 24. No code is executed here.


### Original cell 25

```python
def show_prediction(item):
    m = image_metrics(
        item["mask"],
        item["probability"],
        BEST_THRESHOLD,
    )
    pred = item["probability"] >= BEST_THRESHOLD

    plt.figure(figsize=(16,4))
    plt.subplot(1,4,1)
    plt.imshow(item["image"])
    plt.title(item["stem"])
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(item["mask"],cmap="gray")
    plt.title(f"True: {m['true_coverage']:.1f}%")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(item["probability"],cmap="gray",vmin=0,vmax=1)
    plt.title("Probability")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(pred,cmap="gray")
    plt.title(
        f"Pred: {m['predicted_coverage']:.1f}%\n"
        f"IoU: {m['iou']:.3f}"
    )
    plt.axis("off")
    plt.show()

worst = test_results.sort_values(
    ["iou","coverage_abs_error"],
    ascending=[True,False],
)

display(worst.head(12))

for stem in worst.head(min(12,len(worst)))["stem"]:
    item = next(x for x in test_predictions if x["stem"]==stem)
    show_prediction(item)

```


In [ ]:
# Historical output from original cell 25. No code is executed here.


### Original cell 26

```python
FINAL_MODEL = OUTPUT_DIR/"lichen_coverage_model_v2.keras"
model.save(FINAL_MODEL)

metadata = {
    "image_size":IMAGE_SIZE,
    "threshold":BEST_THRESHOLD,
    "architecture":"MobileNetV2 U-Net",
    "training_images":len(train_df),
    "validation_images":len(val_df),
    "test_images":len(test_df),
    "test_summary":summary,
    "confidence_intervals":cis,
}

with open(OUTPUT_DIR/"model_metadata.json","w") as f:
    json.dump(metadata,f,indent=2)

print("Saved:",FINAL_MODEL)

```


In [ ]:
# Historical output from original cell 26. No code is executed here.


### Original cell 27

```python
from google.colab import files

# Path to your final trained model
final_model_path = OUTPUT_DIR / "lichen_coverage_model_v2.keras"

# Check if the file exists before initiating download
if final_model_path.exists():
    print(f"Downloading {final_model_path.name} to your computer...")
    files.download(str(final_model_path))
else:
    print(f"Error: Model file not found at {final_model_path}. Make sure the training and saving cells have completed successfully.")
```


In [ ]:
# Historical output from original cell 27. No code is executed here.


### Original cell 28

Validation w Apex Imgs


### Original cell 29

```python

import os
import hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import drive, files

drive.mount("/content/drive")

# CHANGE THIS to the newly trained model's actual path.
MODEL_PATH = (
    "/content/drive/MyDrive/EcoSignalCoverage/"
    "training_output/final_retraining_v2/lichen_coverage_model_v2.keras"
)

assert os.path.isfile(MODEL_PATH), f"Not found: {MODEL_PATH}"

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print("Model:", MODEL_PATH)
print("Size (MiB):", round(os.path.getsize(MODEL_PATH) / 1024**2, 2))
print("Input:", model.input_shape)
print("Output:", model.output_shape)
print("Parameters:", model.count_params())

with open(MODEL_PATH, "rb") as f:
    print("SHA256:", hashlib.sha256(f.read()).hexdigest())

assert not isinstance(model.input_shape, list)
assert not isinstance(model.output_shape, list)
assert model.input_shape[-1] == 3
assert model.output_shape[-1] == 1

H, W = model.input_shape[1:3]
assert H is not None and W is not None

```


In [ ]:
# Historical output from original cell 29. No code is executed here.


### Original cell 30

```python

uploaded = files.upload()

image_paths = []

for filename, content in uploaded.items():
    if filename.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        path = os.path.join("/content", os.path.basename(filename))
        with open(path, "wb") as f:
            f.write(content)
        image_paths.append(path)

print("Loaded", len(image_paths), "images")

```


In [ ]:
# Historical output from original cell 30. No code is executed here.


### Original cell 31

```python

THRESHOLD = 0.55  # Change if your NEW model uses another threshold

def validate_image(path, threshold=THRESHOLD):
    original = Image.open(path).convert("RGB")
    original_rgb = np.asarray(original)

    # Match the V2 preprocessing pipeline
    image = tf.convert_to_tensor(original_rgb)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, [H, W], method="bilinear")

    x = image.numpy()

    raw = np.asarray(
        model.predict(x[None, ...], verbose=0)
    )

    prob = np.squeeze(raw)

    if prob.shape != (H, W):
        raise ValueError(f"Unexpected prediction shape: {prob.shape}")

    if not np.isfinite(prob).all():
        raise ValueError("NaN or infinite model output")

    if prob.min() < 0 or prob.max() > 1:
        raise ValueError("Output may contain logits instead of probabilities")

    mask = prob >= threshold

    overlay = x.copy()
    overlay[mask] = (
        0.50 * overlay[mask]
        + 0.50 * np.array([1.0, 0.0, 0.0])
    )

    uncertain = (prob >= 0.4) & (prob <= 0.6)
    confidence = np.maximum(prob, 1 - prob)

    return {
        "original": original_rgb,
        "rgb": x,
        "prob": prob,
        "mask": mask,
        "overlay": np.clip(overlay, 0, 1),
        "coverage": float(mask.mean() * 100),
        "confidence": float(confidence.mean() * 100),
        "uncertain": float(uncertain.mean() * 100),
        "mean_prob": float(prob.mean())
    }


qa_results = {}

for path in image_paths:
    result = validate_image(path)
    qa_results[os.path.basename(path)] = result

    fig, axes = plt.subplots(1, 4, figsize=(19, 5))

    axes[0].imshow(result["rgb"])
    axes[0].set_title("Original Photograph")

    heatmap = axes[1].imshow(
        result["prob"], cmap="viridis", vmin=0, vmax=1
    )
    axes[1].set_title("Lichen Probability")
    fig.colorbar(heatmap, ax=axes[1], fraction=0.046)

    axes[2].imshow(result["mask"], cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Predicted Coverage Mask")

    axes[3].imshow(result["overlay"])
    axes[3].set_title(
        f"Coverage Overlay: {result["coverage"]:.2f}%"
    )

    for ax in axes:
        ax.axis("off")

    plt.suptitle(os.path.basename(path), fontsize=14)
    plt.tight_layout()
    plt.show()

    print("\nImage:", os.path.basename(path))
    print(f"Predicted coverage: {result["coverage"]:.2f}%")
    print(f"Mean probability: {result["mean_prob"]:.4f}")
    print(f"Mean prediction confidence: {result["confidence"]:.2f}%")
    print(f"Uncertain pixels (0.4–0.6): {result["uncertain"]:.2f}%")
    print("-" * 65)

```


In [ ]:
# Historical output from original cell 31. No code is executed here.


### Original cell 32

## CAC stopping benchmark

Treat the model as ready when the untouched test set has:
- mean IoU ≥ 0.75
- median IoU ≥ 0.80
- coverage MAE < 5 percentage points
- at least about 20 test images
- no repeated failure category


### Original cell 33

### `test_summary.json`


### Original cell 34

```python
with open(OUTPUT_DIR / "test_summary.json", "r") as f:
    print(f.read())
```


### Original cell 35

### `test_metrics.csv`


### Original cell 36

```python
with open(OUTPUT_DIR / "test_metrics.csv", "r") as f:
    print(f.read())
```


### Original cell 37

Exports


### Original cell 38

### `test_metrics_by_category.csv`


### Original cell 39

```python
with open(OUTPUT_DIR / "test_metrics_by_category.csv", "r") as f:
    print(f.read())
```


### Original cell 40

### `threshold_sweep.csv`


### Original cell 41

```python
with open(OUTPUT_DIR / "threshold_sweep.csv", "r") as f:
    print(f.read())
```


### Original cell 42

### `confidence_intervals.json`


### Original cell 43

```python
with open(OUTPUT_DIR / "confidence_intervals.json", "r") as f:
    print(f.read())
```


### Original cell 44

### `head_history.csv`


### Original cell 45

```python
with open(OUTPUT_DIR / "head_history.csv", "r") as f:
    print(f.read())
```


### Original cell 46

### `fine_tune_history.csv`


### Original cell 47

```python
with open(OUTPUT_DIR / "fine_tune_history.csv", "r") as f:
    print(f.read())
```


### Original cell 48

### Download Output Files


### Original cell 49

```python
from google.colab import files

# Download test_summary.json
files.download(OUTPUT_DIR / "test_summary.json")
```


### Original cell 50

```python
from google.colab import files

# Download test_metrics.csv
files.download(OUTPUT_DIR / "test_metrics.csv")
```


### Original cell 51

```python
from google.colab import files

# Download test_metrics_by_category.csv
files.download(OUTPUT_DIR / "test_metrics_by_category.csv")
```


### Original cell 52

```python
from google.colab import files

# Download threshold_sweep.csv
files.download(OUTPUT_DIR / "threshold_sweep.csv")
```


### Original cell 53

```python
from google.colab import files

# Download confidence_intervals.json
files.download(OUTPUT_DIR / "confidence_intervals.json")
```


### Original cell 54

```python
from google.colab import files

# Download head_history.csv
files.download(OUTPUT_DIR / "head_history.csv")
```


### Original cell 55

```python
from google.colab import files

# Download fine_tune_history.csv
files.download(OUTPUT_DIR / "fine_tune_history.csv")
```


### Original cell 56

```python
from google.colab import files

files.download(OUTPUT_DIR / "lichen_coverage_model.keras")
```


### Original cell 57

```python
from google.colab import files

files.download(OUTPUT_DIR / "best_model.keras")
```


### Original cell 58

```python
from google.colab import files

files.download(OUTPUT_DIR / "model_metadata.json")
```


### Original cell 59

```python
from google.colab import files

files.download(OUTPUT_DIR / "split_manifest.json")
```


### Original cell 60

```python
from google.colab import files

files.download(OUTPUT_DIR / "test_metrics.csv")
```


### Original cell 61

```python
from google.colab import files

files.download(OUTPUT_DIR / "test_summary.json")
```


### Original cell 62

```python
from google.colab import files

files.download(OUTPUT_DIR / "threshold_sweep.csv")
```
